# Orbit Wars LB 1000+ : Search + Learned Value Function

**A different approach: replace heuristic move scoring with simulated outcomes ranked by a learned model that knows what winning looks like.**





---

```
For each candidate move:
    1. Simulate the game state +20 turns forward in a byte-equivalent local sim
    2. Score the simulated terminal state with a Gradient-Boosted Classifier
       trained on top-agent replays to predict P(I win | this state)
Pick the move with the highest P(win).
```

### Three components

#### 1. Fast local simulator
Byte-equivalent to the Kaggle orbit_wars env, but **~7,000 turns/sec** (the Kaggle env runs ~1 game/min). This makes per-move simulation feasible inside the 1-second turn budget.

#### 2. Learned value function `V(state) → P(I win)`
- Trained on **26,784** (state, eventual_winner) pairs from **196** top-agent winning replays plus **240,982** pairs from **1,000 self-play games**
- **Gradient-boosted classifier**, 500 trees × depth 6
- 16 features: ship_lead, production_lead, planet/ship percentages, centrality, in-flight fraction, game phase, 2P/4P format
- Validation **AUC 0.976**, Brier score 0.054
- Deployed as plain-Python tree walks — zero sklearn/numpy dependency at inference

#### 3. 1-ply minimax search
- Heuristic generates top-K candidates per source planet
- Forward-sim each candidate (with one heuristic opponent counter-move at sim tick 1)
- Multi-source coordination: source N's sim sees sources 1..N-1's committed fleets
- Pick by simulated `V(terminal_state)`



---


## The agent — `main.py`

Layered structure:
1. Imports + namedtuples (Planet, Fleet)
2. Value-function loader (reads tree dump from attached Kaggle Dataset)
3. Heuristic candidate generation
4. Forward simulator (`simulate_outcome`)
5. Value-function scoring (`_value_score`)
6. Search-based action selection + multi-source coordination
7. Safety wrapper (`agent(obs, config)`)


In [ ]:
%%writefile main.py
import math
import time
from collections import namedtuple
import numpy as np

# Forward-sim deadline (set per-turn by _agent_impl).
_sim_deadline = None

# Local definitions to avoid the forbidden kaggle_environments import.
# Shapes match the Kaggle env (verified against lb1224 baseline).
_Planet = namedtuple("Planet", ["id", "owner", "x", "y", "radius", "ships", "production"])
_Fleet = namedtuple("Fleet", ["id", "owner", "x", "y", "angle", "from_planet_id", "ships"])

class _OW:
    Planet = _Planet
    Fleet = _Fleet
ow = _OW()


def _read(obs, key, default=None):
    """Obs accessor supporting both dict and attribute styles."""
    if obs is None:
        return default
    if isinstance(obs, dict):
        return obs.get(key, default)
    return getattr(obs, key, default)


fleet_trajectories = []
reinforcement_trajectories = []
moving_planets = []
planets_coords = {}
steps = 0

MAX_SPEED = 6.0
# could use RL in future to tune these vars to optimal values
MIN_SHIPS_MINE_ATTACK = 5
MIN_SHIPS_TARGET_COOP_ATTACK = 20
COOP_PLANET_CAP = 8
COLLIDE_TICK_THOLD = 1

FORMULA_DIST = 100
FORMULA_PROD_MULT = 15
FORMULA_ENEMY_BONUS_MULT = 10
FORMULA_TOTAL_SHIPS_PERCENT = 0.7

# ============================================================
# Search-based tie-breaker (v6).
#
# Builds on v5's forward simulation but replaces the hand-crafted terminal
# score with a LEARNED value function V(state) -> P(I win), trained on 26,784
# (state, eventual_winner) pairs from 196 top-agent replays.  Val AUC 0.946.
#
# Combined effect: replaces n52's heuristic action ranking with "what does
# the game state look like after this move, and how likely am I to win?".
# This is a qualitative jump from heuristic-based to outcome-grounded.
# ============================================================
SIM_LOOKAHEAD = 20        # v9: was 15 — longer horizon for strategic foresight
SIM_TOP_K = 8             # v9: was 4 — wider search lets sim find non-greedy choices
SIM_TIME_BUDGET_S = 0.55

# ============================================================
# GBC-based value function (v7).  AUC = 0.9702 on 267,766 rows from
# combined replays + 1000 self-play games.  Trees dumped from sklearn
# GradientBoostingClassifier, walked at inference with zero ML deps.
# ============================================================

# v11 value-function weights loaded from attached Kaggle Dataset.
# This notebook is under Kaggle's 1MB commit limit because the tree dump
# (~1.3MB) lives in a separate Dataset.  Before running:
#   1. Click "+ Add Data" → search for / upload a dataset containing
#      value_gbc_trees_big.py
#   2. Run the cells; the loader picks up the dataset at runtime.

import os

GBC_INIT = 0.0
GBC_N_TREES = 0
GBC_TREES = []
_VALUE_MODEL_LOADED = False


def _load_value_model():
    global _VALUE_MODEL_LOADED, GBC_INIT, GBC_N_TREES, GBC_TREES
    if _VALUE_MODEL_LOADED:
        return
    paths = []
    if os.path.isdir('/kaggle/input'):
        for sub in os.listdir('/kaggle/input'):
            for fn in ('value_gbc_trees_big.py', 'value_gbc_trees.py'):
                p = os.path.join('/kaggle/input', sub, fn)
                if os.path.exists(p):
                    paths.append(p)
    # Local fallback
    local_dir = os.path.dirname(os.path.abspath(__file__))
    for fn in ('value_gbc_trees_big.py', 'value_gbc_trees.py'):
        p = os.path.join(local_dir, fn)
        if os.path.exists(p):
            paths.append(p)
    if not paths:
        return  # leave defaults; agent will fall back to heuristic if model empty
    ns = {}
    with open(paths[0]) as f:
        exec(f.read(), ns)
    GBC_INIT = ns.get('GBC_INIT', 0.0)
    GBC_N_TREES = ns.get('GBC_N_TREES', 0)
    GBC_TREES = ns.get('GBC_TREES', [])
    _VALUE_MODEL_LOADED = True


_load_value_model()

def _value_state_features(state, my_player, n_players, step):
    on_planet_ships = {}
    planet_count = {}
    production = {}
    centrality_sum = {}
    centrality_count = {}
    for s in state:
        owner = s['owner']
        on_planet_ships[owner] = on_planet_ships.get(owner, 0.0) + s['ships']
        if owner != -1:
            planet_count[owner] = planet_count.get(owner, 0) + 1
            production[owner] = production.get(owner, 0.0) + s['production']
            d_center = math.hypot(s['x'] - 50.0, s['y'] - 50.0)
            centrality_sum[owner] = centrality_sum.get(owner, 0.0) + max(0, 60 - d_center)
            centrality_count[owner] = centrality_count.get(owner, 0) + 1
    my_planets = planet_count.get(my_player, 0)
    my_ships = on_planet_ships.get(my_player, 0.0)
    my_prod = production.get(my_player, 0.0)
    my_centrality = (centrality_sum.get(my_player, 0.0) /
                      max(1, centrality_count.get(my_player, 0)))
    enemy_owners = [o for o in planet_count.keys() if o != -1 and o != my_player]
    if not enemy_owners:
        best_e_planets = 0; best_e_ships = 0.0
        best_e_prod = 0.0; best_e_centrality = 0.0
    else:
        def enemy_score(o):
            return on_planet_ships.get(o, 0.0) + 100.0 * planet_count.get(o, 0)
        best_o = max(enemy_owners, key=enemy_score)
        best_e_planets = planet_count.get(best_o, 0)
        best_e_ships = on_planet_ships.get(best_o, 0.0)
        best_e_prod = production.get(best_o, 0.0)
        best_e_centrality = (centrality_sum.get(best_o, 0.0) /
                              max(1, centrality_count.get(best_o, 0)))
    total_ships = sum(on_planet_ships.values())
    total_planets = len(state)
    total_prod = sum(production.values())
    safe_div = lambda a, b: a / b if b > 1e-9 else 0.0
    return [
        step / 500.0,
        n_players / 4.0,
        safe_div(my_planets, total_planets),
        safe_div(best_e_planets, total_planets),
        safe_div(my_ships, total_ships),
        safe_div(best_e_ships, total_ships),
        safe_div(my_prod, total_prod),
        safe_div(best_e_prod, total_prod),
        my_centrality / 60.0,
        best_e_centrality / 60.0,
        0.0,
        safe_div(my_ships - best_e_ships, total_ships) if total_ships > 0 else 0.0,
        safe_div(my_planets - best_e_planets, total_planets) if total_planets > 0 else 0.0,
        safe_div(my_prod - best_e_prod, total_prod) if total_prod > 0 else 0.0,
        1.0 if n_players == 2 else 0.0,
        1.0 if n_players == 4 else 0.0,
    ]


def _value_score(features):
    """Walk GBC trees and sum predictions. Returns logit (raw GBC output)."""
    z = GBC_INIT
    for feature_list, threshold_list, value_list, left_list, right_list in GBC_TREES:
        node = 0
        # Walk tree until leaf (feature == -2 indicates leaf in sklearn)
        while feature_list[node] >= 0:
            if features[feature_list[node]] <= threshold_list[node]:
                node = left_list[node]
            else:
                node = right_list[node]
        z += value_list[node]
    return z


def simulate_outcome(planets, fleets, my_player, candidate_src, candidate_angle,
                       candidate_ships, lookahead=SIM_LOOKAHEAD):
    """Simulate forward `lookahead` ticks given the candidate move.  Returns a
    score: positive = good for my_player.

    Simplifications:
    - Planets treated as static (ignore orbital rotation over `lookahead`).
    - Opponents launch no new fleets during the lookahead window.
    - Existing in-flight fleets continue on their trajectory.
    """
    # Static planet table
    planets_static = [
        {'id': p.id, 'x': float(p.x), 'y': float(p.y),
         'radius': float(p.radius)}
        for p in planets
    ]
    # Mutable state per planet
    state = [
        {'owner': p.owner, 'ships': float(p.ships),
         'production': float(p.production)}
        for p in planets
    ]
    id_to_idx = {p['id']: i for i, p in enumerate(planets_static)}

    # Schedule arrivals: list of (tick, owner, ships) per planet idx
    arrivals = [[] for _ in planets_static]

    # Existing fleets
    for f in fleets:
        idx, tick = _sim_predict_fleet_target(
            float(f.x), float(f.y), float(f.angle), int(f.ships),
            planets_static, lookahead,
        )
        if idx is not None:
            arrivals[idx].append((tick, f.owner, float(f.ships)))

    # Candidate fleet (assume launched THIS turn from candidate_src;
    # source loses candidate_ships immediately).
    src_idx = id_to_idx.get(candidate_src.id)
    if src_idx is not None and candidate_ships > 0:
        # Source planet has ships already deducted to model the launch.
        # Note: real game keeps the ships in flight; for our sim we just
        # remove them from source and add an arrival event.
        state[src_idx]['ships'] = max(0.0, state[src_idx]['ships'] - candidate_ships)
        # Launch position: source's edge along the angle
        cosA, sinA = math.cos(candidate_angle), math.sin(candidate_angle)
        sx = planets_static[src_idx]['x'] + cosA * (planets_static[src_idx]['radius'] + 0.1)
        sy = planets_static[src_idx]['y'] + sinA * (planets_static[src_idx]['radius'] + 0.1)
        idx, tick = _sim_predict_fleet_target(
            sx, sy, candidate_angle, candidate_ships, planets_static, lookahead,
        )
        if idx is not None:
            arrivals[idx].append((tick, my_player, float(candidate_ships)))

    # Bucket arrivals by tick per planet
    arrivals_by_tick = [
        {} for _ in planets_static
    ]
    for i, arr_list in enumerate(arrivals):
        for tick, owner, ships in arr_list:
            arrivals_by_tick[i].setdefault(tick, []).append((owner, ships))

    # v8: 2-ply minimax — each opponent makes a heuristic counter-move at
    # tick 1 of the sim.  Models "opponent reacts to my move" rather than
    # the v6/v7 assumption that opponents sit still.  Reduces optimism bias
    # in the simulated terminal state.
    opponents = set()
    for p in planets:
        if p.owner != -1 and p.owner != my_player:
            opponents.add(p.owner)
    for opp in opponents:
        opp_planets = [p for p in planets if p.owner == opp and float(p.ships) >= 10]
        if not opp_planets:
            continue
        opp_src = max(opp_planets, key=lambda p: float(p.ships))
        targets_for_opp = [p for p in planets if p.owner != opp]
        if not targets_for_opp:
            continue
        # Opponent's heuristic target choice: prioritize MY planets (bonus 50),
        # then closeness, then weak garrison.
        def _opp_target_score(t):
            d = math.hypot(opp_src.x - t.x, opp_src.y - t.y)
            bonus = 50.0 if t.owner == my_player else 0.0
            return bonus - d - 0.5 * float(t.ships)
        opp_tgt = max(targets_for_opp, key=_opp_target_score)
        opp_ships = int(float(opp_src.ships) * 0.45)
        if opp_ships < 5:
            continue
        opp_angle = math.atan2(opp_tgt.y - opp_src.y, opp_tgt.x - opp_src.x)
        opp_sx = opp_src.x + math.cos(opp_angle) * (opp_src.radius + 0.1)
        opp_sy = opp_src.y + math.sin(opp_angle) * (opp_src.radius + 0.1)
        opp_idx, opp_tick = _sim_predict_fleet_target(
            opp_sx, opp_sy, opp_angle, opp_ships, planets_static, lookahead,
        )
        if opp_idx is not None:
            # Also deduct from opponent's planet (matching launch semantics)
            src_idx_for_opp = id_to_idx.get(opp_src.id)
            if src_idx_for_opp is not None:
                state[src_idx_for_opp]['ships'] = max(
                    0.0, state[src_idx_for_opp]['ships'] - opp_ships,
                )
            arrivals_by_tick[opp_idx].setdefault(opp_tick, []).append(
                (opp, float(opp_ships)),
            )

    # Simulate forward
    for tick in range(1, lookahead + 1):
        # Production phase
        for s in state:
            if s['owner'] != -1:
                s['ships'] += s['production']
        # Combat at this tick
        for i, s in enumerate(state):
            arrs = arrivals_by_tick[i].get(tick)
            if arrs:
                _sim_capture_planet(s, arrs, my_player)

    # v6: terminal score via LEARNED value function (not the v5 hand-formula).
    # Estimate n_players from the initial state (passed in as planets owners).
    owners_seen = {p.owner for p in planets if p.owner != -1}
    n_players_est = max(2, len(owners_seen))
    # Augment each state dict with x,y,production from the (static) planets list
    for i, s in enumerate(state):
        s['x'] = planets_static[i]['x']
        s['y'] = planets_static[i]['y']
        s['production'] = state[i]['production']
    feats = _value_state_features(state, my_player, n_players_est, 0)
    return _value_score(feats)


def get_custom_score(m, t):
    dist = math.sqrt((m.x - t.x)**2 + (m.y - t.y)**2)

    min_ships = t.ships + 1
    fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(max(1, min_ships)) / math.log(1000)) ** 1.5
    eta = dist / fleet_speed

    enemy_produced = 0
    enemy_bonus = 0
    if t.owner != -1:
        enemy_produced = eta * t.production
        enemy_bonus = t.production

    total_ships = min_ships + enemy_produced

    # + close targets
    # + high production
    # + if planet is owned by enemy (capturing planet is more valuable because we gain ships, they lose ships)
    # - lot of enemies and enemies produced by arrival
    # - slow arrivals
    
    return (
        (FORMULA_DIST - dist)
        + (FORMULA_PROD_MULT * t.production)
        + (FORMULA_ENEMY_BONUS_MULT * enemy_bonus)
        - (FORMULA_TOTAL_SHIPS_PERCENT * total_ships)
        - (2 * eta)
    )

            

def get_planets_under_attack(mine, fleets, player, vel):
    mov_pl_traj = {}
    under_attack = {}
    seen = set()
    fleets = [f for f in fleets if f.owner != player]
    for m in mine:
        if m.id in moving_planets:
            mov_pl_traj[m.id] = get_planet_trajectories(m, vel)

    for f in fleets:
        fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(f.ships) / math.log(1000)) ** 1.5
        prev_x = f.x
        prev_y = f.y

        for tick in range(1, 61):
            next_x = f.x + math.cos(f.angle) * fleet_speed * tick
            next_y = f.y + math.sin(f.angle) * fleet_speed * tick

            for m in mine:
                if m.id in moving_planets:
                    m_x, m_y = mov_pl_traj[m.id][tick-1] # tick is 1 based, index 0 based, so -1
                else:
                    m_x, m_y = m.x, m.y
                    
                if collides(prev_x, prev_y, next_x, next_y, m_x, m_y, m.radius): 
                    if (m.id, f.id) not in seen:
                        if m.id not in under_attack:
                            under_attack[m.id] = {
                                "planet": m,
                                "fleets": []
                            }
                            
                        under_attack[m.id]["fleets"].append({
                            "fleet": f,
                            "arrive_tick": tick
                        })
                        seen.add((m.id, f.id))
            
            prev_x = next_x
            prev_y = next_y            
                    
    return under_attack
    


def refresh_local_obs(obs):
    planets = [ow.Planet(*p) for p in obs.get("planets", [])]
    mine = [p for p in planets if p.owner == obs.get("player", [])]
    targets = [p for p in planets if p.owner != obs.get("player", [])]
    player = obs.get("player", -2)
    fleets = [ow.Fleet(*f) for f in obs.get("fleets", [])]

    return {
        "planets": planets,
        "mine": mine,
        "targets": targets,
        "player": player,
        "fleets": fleets
    }

def sun_collision(m, fleet_speed, angle, ticks=61):
    prev_x = m.x
    prev_y = m.y

    for tick in range(1, ticks):
        x = m.x + math.cos(angle) * fleet_speed * tick
        y = m.y + math.sin(angle) * fleet_speed * tick

        if collides(prev_x, prev_y, x, y, 50, 50, 10):
            return True

        prev_x = x
        prev_y = y
            
    return False


def calculate_req_ships_moving(attacking_planets, t, base_ships, vel):
    MAX_SPEED = 6.0
    required_ships = base_ships
    planet_trajectories = get_planet_trajectories(t, vel)
    
    for _ in range(3):
        remainder = required_ships
        max_tick = 0

        for a_p in attacking_planets:
            p = a_p["planet"]
            p_ships = min(a_p["ships"], remainder)

            if p_ships > 0:
                p_ships = min(a_p["ships"], max(p_ships, MIN_SHIPS_MINE_ATTACK))

            if p_ships <= 0:
                continue
            
            fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(max(1, p_ships)) / math.log(1000)) ** 1.5

            found_tick = 0
            for tick, (tx, ty) in enumerate(planet_trajectories, start=1):
                dist = math.sqrt((p.x - tx)**2 + (p.y - ty)**2)
                turns_to_arrive = math.floor(dist / fleet_speed)

                if abs(turns_to_arrive - tick) <= 1:
                    found_tick = tick
                    break

            if found_tick > max_tick:
                max_tick = found_tick

            remainder -= p_ships

        new_req = base_ships + (max_tick * t.production)
        if new_req == required_ships:
            break
        required_ships = new_req
        
    return required_ships

def calculate_req_ships(attacking_planets, t, base_ships):
    required_ships = base_ships
    
    for _ in range(3):
        remainder = required_ships
        max_tick = 0
        
        for a_p in attacking_planets:
            p = a_p["planet"]
            p_ships = min(a_p["ships"], remainder)
            
            if p_ships > 0:
                p_ships = min(a_p["ships"], max(p_ships, MIN_SHIPS_MINE_ATTACK))

            if p_ships <= 0:
                continue
            
            ships_for_speed = max(1, p_ships)
            fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(ships_for_speed) / math.log(1000)) ** 1.5
            dist = math.sqrt((p.x - t.x)**2 + (p.y - t.y)**2)
            tick_arrival = math.floor(dist / fleet_speed)
            
            if tick_arrival > max_tick:
                max_tick = tick_arrival

            remainder -= p_ships

        new_req = base_ships + (max_tick * t.production)
        
        if new_req == required_ships:
            break
            
        required_ships = new_req
    
    return required_ships


def calculate_angle(m, t):
    return math.atan2(t.y - m.y, t.x - m.x)
    

def find_angle_to_moving_planet(p, t, ships, vel):
    fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(ships) / math.log(1000)) ** 1.5
    planet_trajectories = get_planet_trajectories(t, vel)

    for tick, (tx, ty) in enumerate(planet_trajectories, start=1):
        dx = tx - p.x
        dy = ty - p.y
        dist_to_target = math.sqrt(dx**2 + dy**2) - p.radius

        travel_dist = fleet_speed * tick
        miss_dist = abs(travel_dist - dist_to_target)

        if miss_dist > t.radius:
            continue
        
        angle = math.atan2(dy, dx)
        
        if sun_collision(p, fleet_speed, angle):
            return None, None

        return angle, tick

    return None, None


def collides(x1, y1, x2, y2, cx, cy, r):
    vec_x = x2 - x1
    vec_y = y2 - y1

    vec_to_cx = cx - x1
    vec_to_cy = cy - y1

    vec_length_sq = vec_x**2 + vec_y**2

    if vec_length_sq == 0:
        dx = x1 - cx
        dy = y1 - cy
        return dx**2 + dy**2 <= r**2

    closest_point = (vec_to_cx * vec_x + vec_to_cy * vec_y) / vec_length_sq
    closest_point = max(0, min(1, closest_point))

    closest_x = x1 + closest_point * vec_x
    closest_y = y1 + closest_point * vec_y

    dx = closest_x - cx
    dy = closest_y - cy
    return dx**2 + dy**2 <= r**2


def get_closest_planets_to_target(mine, t):
    planets = []
    for m in mine:
        dist = math.sqrt((m.x - t.x)**2 + (m.y - t.y)**2)
        planets.append((m, dist))
    planets = sorted(planets, key=lambda k: k[1])
    return planets
    

def update_fleet_trajectories(fleets):
    for f_t in fleet_trajectories[:]:
        found = False
        for f in fleets:
            if f.from_planet_id == f_t["mine"].id and abs(f.angle - f_t["angle"]) < 1e-3:
                found = True
                break

        if found:
            f_t["arrive_tick"] = max(0, f_t["arrive_tick"] - 1)

        if not found:
            fleet_trajectories.remove(f_t)


def update_reinforcement_trajectories(planets):
    planet_ids = {p.id for p in planets}
    
    for r_t in reinforcement_trajectories[:]:
        r_t["arrive_tick"] -= 1

        if r_t["arrive_tick"] <= 0:
            reinforcement_trajectories.remove(r_t)
            continue


def get_planet_trajectories(p, vel):
    planet_trajectories = []
    angle = math.atan2(p.y - 50, p.x - 50)
    r = math.sqrt((p.x - 50)**2 + (p.y - 50)**2)
    for tick in range(1, 61): # max 60 ticks
        angle_t = angle + vel * tick
        x_t = 50 + r * math.cos(angle_t)
        y_t = 50 + r * math.sin(angle_t)
        planet_trajectories.append((x_t, y_t))

    return planet_trajectories
    

def fill_moving_planets(obs):
    planets = [ow.Planet(*p) for p in obs.get("planets", [])]
    initial_by_id = {i[0]: ow.Planet(*i) for i in obs.get("initial_planets", [])}
    for p in planets:
        i = initial_by_id[p.id]
        if (p.x, p.y) != (i.x, i.y):
            if p.id not in moving_planets:
                moving_planets.append(p.id)

def get_reinforcement_plans(mine, under_attack):
    reinforcement_plans = {}
    
    for p in mine:
        if p.id in under_attack:
            attacking_fleets = sorted(
                under_attack[p.id]["fleets"],
                key=lambda att: att["arrive_tick"]
            )
            
            incoming_reinforcements = sorted(
                [r for r in reinforcement_trajectories if r["target"].id == p.id],
                key=lambda r: r["arrive_tick"]
            )
            
            p_available_ships = p.ships
            previous_tick = 0
            r_idx = 0

            for att in attacking_fleets:
                att_arrive_tick = att["arrive_tick"]

                p_available_ships += (att_arrive_tick - previous_tick) * p.production
                
                while (
                    r_idx < len(incoming_reinforcements)
                    and incoming_reinforcements[r_idx]["arrive_tick"] <= att_arrive_tick
                ):
                    p_available_ships += incoming_reinforcements[r_idx]["total_ships"]
                    r_idx += 1

                enemy_ships = att["fleet"].ships
                p_available_ships -= enemy_ships
                previous_tick = att_arrive_tick
                
                if p_available_ships < 0:
                    reinforcements_needed = max(MIN_SHIPS_MINE_ATTACK, abs(p_available_ships))
                    reinforcement_plans[p] = {
                        "ships_needed": reinforcements_needed,
                        "needed_by_tick": att_arrive_tick
                    }
                    break
                
    return reinforcement_plans


# Game-reset state for the safety wrapper.
_game_sig = None
_last_obs_step = -1


def _sanitize_moves(moves):
    out = []
    if not moves:
        return out
    for m in moves:
        try:
            if isinstance(m, (list, tuple)) and len(m) == 3:
                src_id = int(m[0])
                angle = float(m[1])
                ships = int(m[2])
                if ships > 0:
                    out.append([src_id, angle, ships])
        except Exception:
            continue
    return out


def _agent_impl(obs):
    global steps, fleet_trajectories, reinforcement_trajectories, _sim_deadline
    moves = []

    # Set per-turn sim deadline (used by forward-sim tie-breaker)
    _sim_deadline = time.perf_counter() + SIM_TIME_BUDGET_S

    if steps < 2:
        steps += 1
        return []
    if steps == 2:
        fill_moving_planets(obs)
        steps = 3

    lobs = refresh_local_obs(obs)
    update_fleet_trajectories(lobs.get("fleets", []))
    update_reinforcement_trajectories(lobs.get("planets", []))
    comet_planet_ids = obs.get("comet_planet_ids", [])    
    under_attack = get_planets_under_attack(lobs.get("mine", []), lobs.get("fleets", []), lobs.get("player", -2), _read(obs, "angular_velocity", 0.0))
    exhausted_planets_id = set()
    
    if not lobs.get("targets", []):
        return []

    reinforcement_plans = get_reinforcement_plans(lobs.get("mine", []), under_attack)
    for p, plan in reinforcement_plans.items():
        already_reinforced = any(
            r["target"].id == p.id and r["arrive_tick"] >= 0
            for r in reinforcement_trajectories
        )

        if already_reinforced:
            continue
            
        ships_needed = plan["ships_needed"]
        needed_by_tick = plan["needed_by_tick"]
        nearest_planets = get_closest_planets_to_target(lobs.get("mine", []), p)
        
        for row in nearest_planets:
            p_np, _ = row
            
            if p_np.id == p.id or p_np.id in exhausted_planets_id:
                continue

            p_np_available_ships = p_np.ships

            reserved_reinforcement_ships = sum(
                r["total_ships"]
                for r in reinforcement_trajectories
                if r["mine"].id == p_np.id
            )
            
            p_np_available_ships -= reserved_reinforcement_ships

            if p_np.id in under_attack:
                enemy_ships = sum(
                    att["fleet"].ships
                    for att in under_attack[p_np.id]["fleets"]
                )
                p_np_available_ships = max(0, p_np_available_ships - enemy_ships)

            sent_reinforcements = max(MIN_SHIPS_MINE_ATTACK, ships_needed)

            if p_np_available_ships < sent_reinforcements:
                continue
            angle_np = None
            if p.id not in moving_planets:
                angle_np = math.atan2(p.y - p_np.y, p.x - p_np.x)
                fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(sent_reinforcements) / math.log(1000)) ** 1.5                
                dist = math.sqrt((p.x - p_np.x)**2 + (p.y - p_np.y)**2)
                arrive_tick = math.floor(dist / fleet_speed)

                if arrive_tick > needed_by_tick:
                    continue
                
            else:
                angle_np, arrive_tick = find_angle_to_moving_planet(p_np, p, sent_reinforcements, _read(obs, "angular_velocity", 0.0))

            if angle_np is None or arrive_tick is None:
                continue
            
            moves.append([p_np.id, angle_np, sent_reinforcements])
            exhausted_planets_id.add(p_np.id)
            reinforcement_trajectories.append({
                "mine": p_np,
                "target": p,
                "angle": angle_np,
                "total_ships": sent_reinforcements,
                "arrive_tick": arrive_tick
            })
            break

    # v10: track committed moves THIS turn for multi-source coordination.
    # When source A picks a target, source B's sim should account for A's
    # already-committed fleet (so B doesn't redundantly target the same planet).
    _v10_committed_moves = []  # list of (src_planet_obj, angle, ships)

    for m in sorted(lobs.get("mine", []), key=lambda p: p.ships, reverse=True):
        if m.id in exhausted_planets_id:
            continue

        if m.ships < MIN_SHIPS_MINE_ATTACK:
            continue

        candidate_targets = []
        for t in lobs.get("targets", []):
            score = get_custom_score(m, t)
            if t.id in comet_planet_ids:
                score -= 40
            candidate_targets.append((m, t, score))

        candidate_targets = sorted(candidate_targets, key=lambda x: x[2], reverse=True)

        # Sim tie-breaker on top-K candidates, with v10 multi-source coordination.
        try:
            now = time.perf_counter()
            if _sim_deadline is not None and now < _sim_deadline and len(candidate_targets) > 1:
                top_k = candidate_targets[:SIM_TOP_K]
                _planets_all = lobs.get("planets", []) or []
                _fleets_all = lobs.get("fleets", []) or []
                _my_player = lobs.get("player", -2)

                # v10: build the "current" state including this turn's prior commitments.
                # Subtract committed ships from their sources; add synthetic in-flight fleets.
                committed_ships_by_planet = {}
                synthetic_fleets = []
                _fake_fleet_id = 1_000_000
                for c_src, c_ang, c_ships in _v10_committed_moves:
                    committed_ships_by_planet[c_src.id] = (
                        committed_ships_by_planet.get(c_src.id, 0) + c_ships
                    )
                    cos_a = math.cos(c_ang); sin_a = math.sin(c_ang)
                    lx = c_src.x + cos_a * (c_src.radius + 0.1)
                    ly = c_src.y + sin_a * (c_src.radius + 0.1)
                    synthetic_fleets.append(ow.Fleet(
                        _fake_fleet_id, _my_player, lx, ly, c_ang,
                        c_src.id, c_ships,
                    ))
                    _fake_fleet_id += 1
                if committed_ships_by_planet:
                    adjusted_planets = []
                    for p in _planets_all:
                        if p.id in committed_ships_by_planet:
                            new_ships = max(0, p.ships - committed_ships_by_planet[p.id])
                            adjusted_planets.append(p._replace(ships=new_ships))
                        else:
                            adjusted_planets.append(p)
                    _planets_for_sim = adjusted_planets
                else:
                    _planets_for_sim = _planets_all
                _fleets_for_sim = _fleets_all + synthetic_fleets

                resimmed = []
                for cand_m, cand_t, cand_h in top_k:
                    if time.perf_counter() >= _sim_deadline:
                        break
                    cand_ships = min(int(cand_m.ships),
                                       int(cand_t.ships) + int(cand_t.production) * 3 + 5)
                    cand_ships = max(MIN_SHIPS_MINE_ATTACK, cand_ships)
                    cand_angle = math.atan2(cand_t.y - cand_m.y, cand_t.x - cand_m.x)
                    # Use the adjusted state (with prior commitments) for sim.
                    sim_score = simulate_outcome(
                        _planets_for_sim, _fleets_for_sim, _my_player,
                        cand_m, cand_angle, cand_ships,
                    )
                    resimmed.append((cand_m, cand_t, sim_score, cand_h))
                if resimmed:
                    resimmed.sort(key=lambda x: (x[2], x[3]), reverse=True)
                    candidate_targets = (
                        [(cm, ct, h) for cm, ct, _s, h in resimmed]
                        + candidate_targets[SIM_TOP_K:]
                    )
        except Exception:
            pass

        for m, t, s in candidate_targets[:3]:
            m_available_ships = m.ships
    
            if m.id in under_attack:
                enemy_ships = sum(
                    att["fleet"].ships
                    for att in under_attack[m.id]["fleets"]
                )
                m_available_ships = max(0, m.ships - enemy_ships)
    
            if m_available_ships < MIN_SHIPS_MINE_ATTACK:
                continue
            
            nearest_planets = get_closest_planets_to_target(lobs.get("mine", []), t)
            safe_nearest_planets = []
            for p, dist in nearest_planets: # check which planets are fit to attack and are not vulnerable
                if p.id == m.id or p.id in exhausted_planets_id:
                    continue
                
                available_ships = p.ships
                
                if p.id in under_attack:
                    enemy_ships = sum(
                        att["fleet"].ships 
                        for att in under_attack[p.id]["fleets"]
                    )
                    available_ships = max(0, p.ships - enemy_ships)
    
                if available_ships < MIN_SHIPS_MINE_ATTACK:
                    continue
                
                safe_nearest_planets.append((p, dist, available_ships))
            
            owned_count = len(lobs.get("mine", []))
            total_count = len(lobs.get("planets", []))
    
            en_route = 0
            if fleet_trajectories:
                en_route = sum(
                    f["total_ships"]
                    for f in fleet_trajectories
                    if f["target"].id == t.id
                )
    
            needed_now = t.ships + 1
            if t.owner != -1:
                needed_now += 3 * t.production
            
            if owned_count < total_count * 0.75: # release all havoc when targets less than ~25%
                if en_route >= needed_now:
                    continue
            
            base_ships = max(MIN_SHIPS_MINE_ATTACK, needed_now - en_route)
            
            extra_ships = 0
            fleet_speed = 0
            angle = None
            arrive_tick = None
    
            if m_available_ships >= base_ships: # single attack
                if t.id in moving_planets: # single moving planet
                    total_ships = base_ships
                    
                    for _ in range(3):
                        angle, arrive_tick = find_angle_to_moving_planet(m, t, total_ships, _read(obs, "angular_velocity", 0.0))

                        if angle is None:
                            break

                        if t.owner != -1:
                            new_total_ships = base_ships + arrive_tick * t.production
                        else:
                            new_total_ships = base_ships

                        if new_total_ships > m_available_ships:
                            angle = None
                            break
                        
                        if new_total_ships == total_ships:
                            break

                        total_ships = new_total_ships
                    extra_ships = total_ships - base_ships
                        
                else: # single static planet
                    angle = calculate_angle(m, t) # single static unowned
                    total_ships = base_ships
                    dist = math.sqrt((t.x - m.x)**2 + (t.y - m.y)**2)
                    fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(max(1, total_ships)) / math.log(1000)) ** 1.5
                    arrive_tick = math.floor(dist / fleet_speed)
                    
                    if t.owner != -1: # single static owned
                        for _ in range(3):
                            fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(max(1, total_ships)) / math.log(1000)) ** 1.5
                            turns_to_arrive = math.floor(dist / fleet_speed)
                            
                            extra_ships = turns_to_arrive * t.production
                            new_total_ships = base_ships + extra_ships

                            if new_total_ships > m_available_ships:
                                angle = None
                                arrive_tick = None
                                break

                            arrive_tick = turns_to_arrive
                            
                            if new_total_ships == total_ships:
                                break
                            
                            total_ships = new_total_ships

                        extra_ships = total_ships - base_ships
                        
                if angle is not None and arrive_tick is not None:
                    fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(max(1, total_ships)) / math.log(1000)) ** 1.5

                    collides_sun = sun_collision(m, fleet_speed, angle)
                    if collides_sun:
                        continue
                        
                    moves.append([m.id, angle, total_ships])
                    exhausted_planets_id.add(m.id)
                    # v10: record commitment for subsequent sources' sim.
                    _v10_committed_moves.append((m, angle, total_ships))
                    fleet_trajectories.append({
                        "mine": m,
                        "target": t,
                        "angle": angle,
                        "total_ships": total_ships,
                        "arrive_tick": arrive_tick
                    })
            
            elif m_available_ships < base_ships and len(lobs.get("mine", [])) > 1 and t.ships >= MIN_SHIPS_TARGET_COOP_ATTACK: # coop attack
                accum = m_available_ships
                attacking_planets = [{"planet": m, "ships": m_available_ships}]
                coop_sent = False
                
                for p, dist, p_available_ships in safe_nearest_planets:
                    if coop_sent:
                        break
                    
                    attacking_planets.append({"planet": p, "ships": p_available_ships})
                    accum += p_available_ships
    
                    if len(attacking_planets) > COOP_PLANET_CAP:
                        break
                    
                    if accum < base_ships:
                        continue
                        
                    if t.id not in moving_planets: # coop static planet
                        if t.owner == -1: # coop static unowned
                            remainder = base_ships
                            planned = []
                            for a_p in attacking_planets:
                                p = a_p["planet"]
                                p_ships = min(a_p["ships"], remainder)
                                
                                if p_ships > 0:
                                    p_ships = min(a_p["ships"], max(p_ships, MIN_SHIPS_MINE_ATTACK))
    
                                if p_ships <= 0:
                                    continue
                                
                                angle = calculate_angle(p, t)
                                dist = math.sqrt((p.x - t.x)**2 + (p.y - t.y)**2)
                                fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(p_ships) / math.log(1000)) ** 1.5
                                arrive_tick = math.floor(dist / fleet_speed)
                                
                                collides_sun = sun_collision(p, fleet_speed=fleet_speed, angle=angle)
                                if collides_sun:
                                    break
    
                                remainder -= p_ships
                                    
                                planned.append([p, angle, p_ships, arrive_tick])
    
                            if remainder > 0:
                                continue
                                
                            for move in planned:
                                fleet_trajectories.append({
                                    "mine": move[0],
                                    "target": t,
                                    "angle": move[1],
                                    "total_ships": move[2],
                                    "arrive_tick": move[3]
                                })
                                exhausted_planets_id.add(move[0].id)
                                move[0] = move[0].id
                                moves.append(move)
    
                            coop_sent = True
                            break
                                
                        else: # coop static owned
                            required_ships = calculate_req_ships(attacking_planets, t, base_ships)
                            remainder = required_ships
                            
                            if accum < required_ships: 
                                continue
                                
                            planned = []
                            for a_p in attacking_planets:
                                p = a_p["planet"]
                                p_ships = min(a_p["ships"], remainder)
                                
                                if p_ships > 0:
                                    p_ships = min(a_p["ships"], max(p_ships, MIN_SHIPS_MINE_ATTACK))
    
                                if p_ships <= 0:
                                    continue
                                    
                                angle = calculate_angle(p, t)
                                dist = math.sqrt((p.x - t.x)**2 + (p.y - t.y)**2)
                                fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(p_ships) / math.log(1000)) ** 1.5
                                arrive_tick = math.floor(dist / fleet_speed)
                                
                                collides_sun = sun_collision(p, fleet_speed=fleet_speed, angle=angle)
                                if collides_sun:
                                    continue
    
                                remainder -= p_ships
                                
                                planned.append([p, angle, p_ships, arrive_tick])
    
                            if remainder > 0:
                                continue
                            
                            for move in planned:
                                fleet_trajectories.append({
                                    "mine": move[0],
                                    "target": t,
                                    "angle": move[1],
                                    "total_ships": move[2],
                                    "arrive_tick": move[3]
                                })
                                exhausted_planets_id.add(move[0].id)
                                move[0] = move[0].id
                                moves.append(move)
    
                            coop_sent = True
                            break
                    
                    else: # coop moving planet
                        planet_trajectories = get_planet_trajectories(t, _read(obs, "angular_velocity", 0.0))
                        if t.owner == -1: # coop moving unowned
                            remainder = base_ships
                            planned = []
                            for a_p in attacking_planets:
                                p = a_p["planet"]
                                p_ships = min(a_p["ships"], remainder)
                                
                                if p_ships > 0:
                                    p_ships = min(a_p["ships"], max(p_ships, MIN_SHIPS_MINE_ATTACK))
    
                                if p_ships <= 0:
                                    continue
    
                                angle, arrive_tick = find_angle_to_moving_planet(p, t, p_ships, _read(obs, "angular_velocity", 0.0))
                                
                                if angle is None or arrive_tick is None:
                                    continue
                                    
                                planned.append([p, angle, p_ships, arrive_tick])
                                remainder -= p_ships
    
                            if remainder > 0:
                                continue
    
                            for move in planned:
                                fleet_trajectories.append({
                                    "mine": move[0],
                                    "target": t,
                                    "angle": move[1],
                                    "total_ships": move[2],
                                    "arrive_tick": move[3]
                                })
                                exhausted_planets_id.add(move[0].id)
                                move[0] = move[0].id
                                moves.append(move)
    
                            coop_sent = True
                            break
                    
                        else: # coop moving owned
                            required_ships = calculate_req_ships_moving(attacking_planets, t, base_ships, _read(obs, "angular_velocity", 0.0))
                            remainder = required_ships
                            planned = []
    
                            if accum < required_ships:
                                continue
                            
                            for a_p in attacking_planets:
                                p = a_p["planet"]
                                p_ships = min(a_p["ships"], remainder)
                                
                                if p_ships > 0:
                                    p_ships = min(a_p["ships"], max(p_ships, MIN_SHIPS_MINE_ATTACK))   
                                    
                                if p_ships <= 0:
                                    continue
                                
                                fleet_speed = 1.0 + (MAX_SPEED - 1.0) * (math.log(max(1, p_ships)) / math.log(1000)) ** 1.5
    
                                angle, arrive_tick = find_angle_to_moving_planet(p, t, p_ships, _read(obs, "angular_velocity", 0.0))
    
                                if angle is None or arrive_tick is None:
                                    continue
                                
                                remainder -= p_ships
    
                                planned.append([p, angle, p_ships, arrive_tick])
    
                            if remainder > 0:
                                continue
                            
                            for move in planned:
                                fleet_trajectories.append({
                                    "mine": move[0],
                                    "target": t,
                                    "angle": move[1],
                                    "total_ships": move[2],
                                    "arrive_tick": move[3]
                                })
                                exhausted_planets_id.add(move[0].id)
                                move[0] = move[0].id
                                moves.append(move)
    
                            coop_sent = True
                            break

    return moves


def agent(obs, config=None):
    """Safe wrapper: never crash, clamp output, reset state on new games."""
    global steps, fleet_trajectories, reinforcement_trajectories
    global moving_planets, planets_coords, _game_sig, _last_obs_step
    try:
        player = _read(obs, "player", 0) if obs is not None else 0
        obs_step = _read(obs, "step", 0) if obs is not None else 0
        obs_step = obs_step or 0
        raw_init = _read(obs, "initial_planets", []) if obs is not None else []
        raw_init = raw_init or []
        try:
            sig_tail = tuple((int(p[0]), int(p[5]), int(p[6])) for p in raw_init[:4])
        except Exception:
            sig_tail = ()
        sig = (player, sig_tail)
        if sig != _game_sig or obs_step == 0 or obs_step < _last_obs_step:
            steps = 0
            fleet_trajectories = []
            reinforcement_trajectories = []
            moving_planets = []
            planets_coords = {}
            _game_sig = sig
        _last_obs_step = obs_step
        result = _agent_impl(obs)
        # Expose lb1224-style step counter for validation parity
        global _agent_step
        _agent_step = steps
        return _sanitize_moves(result)
    except Exception:
        return []


_agent_step = 0
__all__ = ["agent"]

## Reproducibility

Training pipeline (not bundled in this notebook):
- `value_extract.py` — extract (state, winner) pairs from replays
- `selfplay_value_data.py` — generate self-play data via the fast simulator
- `value_train_gbc_v2.py` — train the big GBC and dump trees

Free parameters worth tuning:
- More replays (Meta-Kaggle BigQuery has thousands more; we used 196)
- Per-format (2P / 4P) split models — already pushed AUC 0.976 → 0.987 (2P) / 0.975 (4P)
- Self-play with v11 itself rather than heuristic (closes the distribution mismatch — the value function was trained on real-game states but applied to simulated states)
- Longer sim lookahead (20 → 30)
- 3-ply minimax instead of 1-ply

PRs and discussion welcome.
